In [ ]:
!pip3 install ktrain

     |████████████████████████████████| 25.2MB 121kB/s 
     |████████████████████████████████| 421.8MB 30kB/s 
     |████████████████████████████████| 6.7MB 31.7MB/s 
     |████████████████████████████████| 983kB 48.0MB/s 
     |████████████████████████████████| 245kB 49.7MB/s 
     |████████████████████████████████| 675kB 35.9MB/s 
     |████████████████████████████████| 471kB 49.8MB/s 
     |████████████████████████████████| 450kB 48.2MB/s 
     |████████████████████████████████| 3.9MB 51.1MB/s 
     |████████████████████████████████| 1.1MB 54.6MB/s 
     |████████████████████████████████| 3.8MB 53.6MB/s 
     |████████████████████████████████| 890kB 54.4MB/s 
  Created wheel for ktrain: filename=ktrain-0.17.3-cp36-none-any.whl size=25249269 sha256=c48df9bc77481647dfccd447f0e07c74240019e8314ff7dee349d744fe890dd4
  Stored in directory: /root/.cache/pip/wheels/24/87/78/8dcbc028acc22bc106e03a954f0de2f02e57fb42438ecb177d
  Created wheel for keras-bert: filename=keras_bert-0.84.0-cp36-no

In [ ]:
import ktrain
import pandas as pd
from ktrain import text
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
data = pd.read_csv('LabeledNews.csv')
data.head()

,News,Sentiment
0,"Ferrari Profit Acceleration Eases A Bit, As Ma...",Negative
1,Automatic Data Processing (ADP) Earnings Expec...,Positive
2,AirAsia CEO Fernandes and chairman step aside ...,Negative
3,"Mexico's auto industry restart gathers steam, ...",Neutral
4,Investors eye consumer discretionary stocks as...,Neutral


In [ ]:
data.Sentiment.value_counts()

Positive    1265
Neutral     1265
Negative    1265
Name: Sentiment, dtype: int64

In [ ]:
txt = data.News.to_list()
sent = data.Sentiment.to_list()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(txt, sent, test_size = 0.20, random_state = 0)

In [ ]:
(x_train,  y_train), (x_test, y_test), preproc = text.texts_from_array(x_train=X_train, y_train=y_train,
                                                                       x_test=X_test, y_test=y_test,
                                                                       class_names=["Negative", "Neutral", "Positive"],
                                                                       maxlen=350,
                                                                       preprocess_mode='bert')

task: text classification
downloading pretrained BERT model (uncased_L-12_H-768_A-12.zip)...
[██████████████████████████████████████████████████]
extracting pretrained BERT model...
done.

cleanup downloaded zip...
done.

preprocessing train...
language: en


Is Multi-Label? False
preprocessing test...
language: en


/usr/local/lib/python3.6/dist-packages/ktrain/text/preprocessor.py:520: UserWarning: class_names argument was ignored, as they were extracted from string labels in dataset
  if self.get_classes(): warnings.warn('class_names argument was ignored, as they were extracted from string labels in dataset')


In [ ]:
model = text.text_classifier(name = 'bert', train_data=(x_train, y_train), preproc=preproc)

Is Multi-Label? False
maxlen is 350
done.


In [ ]:
learner = ktrain.get_learner(model, train_data=(x_train, y_train), batch_size=6)

In [ ]:
learner.fit_onecycle(2e-5, 4)



begin training using onecycle policy with max lr of 2e-05...
Train on 3036 samples
Epoch 1/4
3036/3036 [==============================] - 381s 125ms/sample - loss: 0.8245 - accuracy: 0.6133
Epoch 2/4
3036/3036 [==============================] - 368s 121ms/sample - loss: 0.4049 - accuracy: 0.8458
Epoch 3/4
3036/3036 [==============================] - 371s 122ms/sample - loss: 0.1980 - accuracy: 0.9312
Epoch 4/4
3036/3036 [==============================] - 368s 121ms/sample - loss: 0.0636 - accuracy: 0.9816


In [ ]:
learner.validate(val_data=(x_test, y_test), class_names=["Negative", "Neutral", "Positive"])

              precision    recall  f1-score   support

    Negative       0.92      0.90      0.91       240
     Neutral       0.84      0.80      0.82       275
    Positive       0.82      0.89      0.85       244

    accuracy                           0.86       759
   macro avg       0.86      0.86      0.86       759
weighted avg       0.86      0.86      0.86       759



array([[215,  19,   6],
       [ 14, 221,  40],
       [  5,  23, 216]])

In [ ]:
predictor = ktrain.get_predictor(learner.model, preproc)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


In [ ]:
predictor.save('/content/drive/My Drive/BertPredictorForNewsSentimentClassifier')

In [ ]:
reloaded_predictor = ktrain.load_predictor('/content/drive/My Drive/BertPredictorForNewsSentimentClassifier')

In [ ]:
reloaded_predictor.predict('''A number of companies in the United States have decided to shutter their retail outlets and scale
                             back on business in cities most disrupted by the protests organized in the wake of George Floyd's death''')

'Negative'